In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

feature_df = pd.read_pickle("/content/drive/MyDrive/SmartRail/features_full.pkl")

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

X = feature_df.drop(columns=["failure"])
y = feature_df["failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Setup done")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup done


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

single_tree = DecisionTreeClassifier(class_weight='balanced', max_depth=10, random_state=42)
single_tree.fit(X_train, y_train)
auc_single = roc_auc_score(y_test, single_tree.predict_proba(X_test)[:, 1])

bagging = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=10),
                              n_estimators=20, random_state=42, n_jobs=-1)
bagging.fit(X_train, y_train)
auc_bagging = roc_auc_score(y_test, bagging.predict_proba(X_test)[:, 1])

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                              max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

print(f"Single Tree AUC: {auc_single:.4f}")
print(f"Bagging AUC: {auc_bagging:.4f}")
print(f"Random Forest AUC: {auc_rf:.4f}")

Single Tree AUC: 0.9991
Bagging AUC: 0.9999
Random Forest AUC: 1.0000


In [ ]:
joblib.dump(single_tree, "/content/drive/MyDrive/SmartRail/single_tree.pkl")
joblib.dump(bagging, "/content/drive/MyDrive/SmartRail/bagging.pkl")
joblib.dump(rf, "/content/drive/MyDrive/SmartRail/random_forest.pkl")

with open("/content/drive/MyDrive/SmartRail/results.json", "r") as f:
    results = json.load(f)

results["Bagging"] = {"auc": 0.9999}
results["Random Forest"] = {"auc": 1.0000}

with open("/content/drive/MyDrive/SmartRail/results.json", "w") as f:
    json.dump(results, f)

print("Saved")

Saved


In [ ]:
from sklearn.ensemble import AdaBoostClassifier

adaboost = AdaBoostClassifier(n_estimators=100, random_state=42)
adaboost.fit(X_train, y_train)

auc_ada = roc_auc_score(y_test, adaboost.predict_proba(X_test)[:, 1])
print(f"AdaBoost AUC: {auc_ada:.4f}")

joblib.dump(adaboost, "/content/drive/MyDrive/SmartRail/adaboost.pkl")

AdaBoost AUC: 0.9999


['/content/drive/MyDrive/SmartRail/adaboost.pkl']

In [ ]:
with open("/content/drive/MyDrive/SmartRail/results.json", "r") as f:
    results = json.load(f)

results["AdaBoost"] = {"auc": 0.9999}

with open("/content/drive/MyDrive/SmartRail/results.json", "w") as f:
    json.dump(results, f)

print("Saved")

Saved
